# 🌀 **Synoptic-Scale Forecasting**

---

### **Context**
In the previous notebooks, situational awareness was established through surface weather analysis, synoptic-scale diagnosis, and column-scale thermodynamic profiling using observational data and model fields. Here, we transition to an operational forecasting workflow by retrieving deterministic and ensemble forecast products for a configurable target domain. The former yields a single forecast trajectory, whereas the latter uses an ensemble of perturbed model integrations to quantify forecast uncertainty and extract probabilistic information. The applied workflow advances from deterministic synoptic-scale forecasting with the Global Forecast System (GFS), through ensemble mean and spread evaluation with the Global Ensemble Forecast System (GEFS), to point-based ensemble meteograms and compound probability diagnostics assessing the risk of high-impact weather events.

### **Learning Goals:**
- 🎯 **Goal 1 | Upper-Air Deterministic Forecast**: Construct animated four-panel upper-air charts from Global Forecast System (GFS) output to diagnose synoptic-scale weather development
- 🎯 **Goal 2 | Ensemble Mean and Spread**: Compute and interpret the ensemble mean and standard deviation fields for mean sea-level pressure (MSLP) from Global Ensemble Forecast System (GEFS) output to diagnose spatial forecast uncertainty
- 🎯 **Goal 3 | Point-Based Ensemble Forecast**: Construct a three-variable ensemble meteogram for a specific location and characterize the temporal evolution of forecast uncertainty
- 🎯 **Goal 4 | Weather Impact Assessment**: Construct a compound probability product by combining multiple thresholded ensemble variables, and interpret the spatial output to diagnose high-impact weather potential

**Run the notebook via the Binder platform:**

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/one-weather-lab/weather-analysis-and-forecasting/HEAD?urlpath=notebooks/notebooks/05_synoptic_forecasting.ipynb)

## ⚙️ Setup, Configuration, and Data Acquisition

The three cells below execute the full setup, configuration, and data acquisition pipeline.  

First, the necessary libraries are imported and the environment is configured. 

Then, the target date and hour, domain, point of interest, and forecast parameters are configured following the instructions below:

1. **Choose your Data Mode** ('realtime' for the most recently completed GFS/GEFS cycle or 'retrospective' for a specific historical cycle)
2. **Adjust the forecast parameters** to set the forecast horizon and time step, and number of GEFS ensemble members
3. **Set your Point of Interest** by specifying a location name, latitude, and longitude 
4. **Set your target date and cycle hour** (only applicable for 'retrospective' mode; cycle hour must be 0, 6, 12, or 18 UTC)
5. **Set the weather impact scenario** to activate the corresponding compound probability thresholds ('Convection', 'Windstorm', 'Heat wave', or 'Cold outbreak')

Finally, GFS deterministic forecast fields and GEFS ensemble member data are downloaded and stored locally.

> **Note on impact thresholds**: The thresholds applied here are illustrative, chosen to demonstrate the compound probability framework rather than to replicate operational warning criteria. In practice, threshold derivation, based on [World Meteorological Organization guidelines](https://etrp.wmo.int/pluginfile.php/42254/mod_page/content/18/WMO-1150_multihazard-guidelines_en.pdf), follows rigorous procedures ranging from methods grounded in direct input from representatives of the affected sector to impact modeling that integrates meteorological, vulnerability, and exposure data

> **Note on data sources**: GFS and GEFS data available via the Herbie interface spans two archival periods with different file formats and resolutions. 
> - Post-2021 (NOMADS/AWS): 0.25° spatial resolution. Single-variable subset GRIBs permit selective downloads for targeted analysis.
> - Pre-2021 (NCEI archive): 0.5° spatial resolution. Data stored in flat dated directories (YYYYMMDD/) containing full-field GRIBs.

> **Note on unexpected kernel restart**: During execution of data-download cells, the kernel may restart unexpectedly due to memory constraints. In that case, execute again all cells sequentially from code cell 1 (Setup) onward.

In [ ]:
# Import required libraries
from __future__ import annotations

import sys
import time as _time
import warnings
from pathlib import Path

# Core data science stack
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import Image, display

# Add local utils to path
sys.path.insert(0, str(Path('../utils').resolve()))

# Import custom utilities
from herbie_gfs import (
    fetch_gfs_analysis,
    fetch_gfs_forecast, resolve_gfs_data, build_gfs_forecast_ds,
)
from herbie_gefs import (
    fetch_gefs_members, resolve_gefs_data, build_gefs_forecast_da,
)
from gfs_diagnostics import (
    compute_temperature_advection,
    compute_relative_vorticity,
    compute_wind_speed,
)
from meteogram_helpers import (
    get_nearest_grid_point,
    build_meteogram,
)
from plot_helpers import (
    plot_ensemble_mean_spread,
    plot_four_panel_forecast_animation,
    plot_compound_probability_animation,
)

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Set up nice plotting defaults
plt.rcParams['figure.figsize'] = [10, 8]
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11

print('[OK]   All libraries loaded successfully!')

In [ ]:
# =============================================================================
# CONFIGURATION SECTION
# =============================================================================

# ---------------------------------------------------------------------------
# DATA MODE
# ---------------------------------------------------------------------------
# 'realtime'      -> target the most recently completed GFS/GEFS cycle.
#
# 'retrospective' -> fetch a specific historical cycle defined by
#                    TARGET_DATE and TARGET_HOUR.
DATA_MODE = 'retrospective'   # <-- CHANGE TO 'realtime' FOR LATEST FORECAST

# ---------------------------------------------------------------------------
# RETROSPECTIVE SETTINGS (used only when DATA_MODE = 'retrospective')
# ---------------------------------------------------------------------------
# Default: Halkidiki convective outbreak (10 July 2019, 00Z).
TARGET_DATE = '2023-07-21'   # <-- MODIFY THIS (YYYY-MM-DD)
TARGET_HOUR =  6              # <-- GFS/GEFS cycle hour: 0 | 6 | 12 | 18

# ---------------------------------------------------------------------------
# FORECAST HORIZON
# ---------------------------------------------------------------------------
FORECAST_HORIZON_H = 48          # <-- forecast horizon (hours; GFS/GEFS max: 384)
FORECAST_STEP_H   = 6           # <-- forecast time step (hours; GFS/GEFS min: 3)

# ---------------------------------------------------------------------------
# ENSEMBLE MEMBERS
# ---------------------------------------------------------------------------
N_MEMBERS = 10   # <-- number of GEFS members to download (max 31: 1 control + 30 perturbed)

# ---------------------------------------------------------------------------
# PARALLEL DOWNLOAD
# ---------------------------------------------------------------------------
MAX_WORKERS = 8   # <-- parallel download threads (GFS + GEFS fetch)

# ---------------------------------------------------------------------------
# TARGET DOMAIN (default: European domain)
# ---------------------------------------------------------------------------
LON_MIN, LON_MAX = -25, 45
LAT_MIN, LAT_MAX =  30, 72

# ---------------------------------------------------------------------------
# POINT OF INTEREST (POI)
# ---------------------------------------------------------------------------
POI_NAME = 'Ioannina, Greece'    # <-- MODIFY for a different location
POI_LAT  =  39.67                # <-- latitude  (degrees North)
POI_LON  =  20.85                # <-- longitude (degrees East)

# ---------------------------------------------------------------------------
# DATA DIRECTORY
# ---------------------------------------------------------------------------
DATA_DIR = Path('data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------------
# CASE DEFINITIONS
# ---------------------------------------------------------------------------
# Maps each impact scenario to its threshold variables and values.
CASE_DEFINITIONS = {
    'Convection': {
        'variables'  : ['cape', 'cin'],
        'thresholds' : {'cape': 1000.0, 'cin': -60.0},
        'units'      : {'cape': 'J/kg', 'cin': 'J/kg'},
    },
    'Heat wave': {
        'variables'  : ['tmax_2m', 'tmin_2m'],
        'thresholds' : {'tmax_2m': 40.0, 'tmin_2m': 25.0},
        'units'      : {'tmax_2m': 'degC', 'tmin_2m': 'degC'},
    },
    'Windstorm': {
        'variables'  : ['gust10m', 'precip_6h'],
        'thresholds' : {'gust10m': 60.0, 'precip_6h': 20.0},
        'units'      : {'gust10m': 'km/h', 'precip_6h': 'mm'},
    },
    'Cold outbreak': {
        'variables'  : ['tmin_2m', 'snod'],
        'thresholds' : {'tmin_2m': 0.0, 'snod': 0.01},
        'units'      : {'tmin_2m': 'degC', 'snod': 'm'},
    },
}

ACTIVE_CASE = 'Heat wave'   # <-- MODIFY THIS to change the compound probability target
CASE_CFG    = CASE_DEFINITIONS[ACTIVE_CASE]

# =============================================================================
if DATA_MODE == 'retrospective':
    _valid_str = f'{TARGET_DATE} {TARGET_HOUR:02d}:00 UTC'
else:
    _valid_str = 'latest available cycle'

print(f'[DATA MODE] {DATA_MODE.upper()} | target: {_valid_str}')
print(f'[HORIZON  ] {FORECAST_HORIZON_H} h  |  forecast step {FORECAST_STEP_H} h')
print(f'[N_MEMBERS] {N_MEMBERS}')
print(f'[WORKERS  ] {MAX_WORKERS}')
print(f'[DOMAIN   ] lon {LON_MIN} to {LON_MAX} deg  lat {LAT_MIN} to {LAT_MAX} deg')
print(f'[POI      ] {POI_NAME}  ({POI_LAT:.2f} N, {POI_LON:.2f} E)')
print(f'[CASE     ] {ACTIVE_CASE}')

In [ ]:
# =============================================================================
# DATA DOWNLOAD
# =============================================================================

# ---------------------------------------------------------------------------
# GFS deterministic forecast
# ---------------------------------------------------------------------------
gfs_cycle, _gfs_stored = resolve_gfs_data(
    DATA_MODE, TARGET_DATE, TARGET_HOUR, FORECAST_HORIZON_H, FORECAST_STEP_H, DATA_DIR,
)
print(f'[GFS CYCLE] {gfs_cycle.strftime("%Y-%m-%d %H:%M UTC")}')
print(f'[GFS FILES ] {"Stored locally" if _gfs_stored else "Not stored locally"}')
if not _gfs_stored:
    print('[...]   Fetching GFS forecast data')
    _start = _time.time()
    fetch_gfs_forecast(
        mode=DATA_MODE, target_date=TARGET_DATE, target_hour=TARGET_HOUR,
        horizon_h=FORECAST_HORIZON_H, step_h=FORECAST_STEP_H, data_dir=DATA_DIR,
        max_workers=MAX_WORKERS,
    )
    print(f'[TIME  ] {(_time.time() - _start):.1f} s')

# ---------------------------------------------------------------------------
# GEFS ensemble forecast
# ---------------------------------------------------------------------------
_case_vars = list(CASE_CFG['variables'])

_gefs_vars_to_fetch = list(dict.fromkeys(
    ['mslp', 't2m', 'precip_6h', 'gust10m'] + _case_vars
))

_fetch_kwargs = dict(
    mode=DATA_MODE,
    target_date=TARGET_DATE if DATA_MODE == 'retrospective' else None,
    target_hour=TARGET_HOUR if DATA_MODE == 'retrospective' else None,
    horizon_h=FORECAST_HORIZON_H,
    data_dir=DATA_DIR,
)

_gefs_cycle = None
for _var in _gefs_vars_to_fetch:
    _cycle, _gefs_stored = resolve_gefs_data(
        DATA_MODE,
        TARGET_DATE if DATA_MODE == 'retrospective' else None,
        TARGET_HOUR if DATA_MODE == 'retrospective' else None,
        horizon_h=FORECAST_HORIZON_H,
        data_dir=DATA_DIR,
        variable=_var,
    )
    if _gefs_cycle is None:
        _gefs_cycle = _cycle
        print(f'[GEFS CYCLE] {_gefs_cycle.strftime("%Y-%m-%d %H:%M UTC")}')
    if not _gefs_stored:
        print(f'[...]   Fetching GEFS {_var}')
        _start = _time.time()
        fetch_gefs_members(**_fetch_kwargs, variable=_var, n_members=N_MEMBERS, max_workers=MAX_WORKERS)
        print(f'[TIME  ] {(_time.time() - _start):.1f} s')
    else:
        print(f'[GEFS FILES ] GEFS {_var}: stored locally')

print('[OK]   All data downloads complete.')

# Section 1: Upper-Air Deterministic Forecast

A deterministic forecast is based on a single numerical weather prediction (NWP) run, estimating how the atmospheric state will evolve from the initialization time through the forecast horizon. In this section, we use the GFS deterministic forecast to assess the evolution of synoptic-scale upper-air features.

The four diagnostic levels established in the synoptic analysis notebooks serve as the forecast panels:

- Τhe **850 hPa temperature** field to monitor lower-tropospheric thermal structure
- Τhe **700 hPa relative humidity** field to assess how moisture availability for cloud formation and precipitation evolves
- Τhe **500 hPa geopotential height** field to track the development of upper-level troughs and ridges
- The **250 hPa wind speed** field to follow the jet stream and embedded streaks

## 1.1 GFS Deterministic Forecast Fields

The cell below builds GFS forecast fields for the four diagnostic levels, combining all lead times into a single dataset cropped to the analysis domain.

In [ ]:
# Build xarray Dataset from locally stored GFS forecast GRIB files
_start = _time.time()
gfs_ds = build_gfs_forecast_ds(
    cycle=gfs_cycle,
    horizon_h=FORECAST_HORIZON_H,
    step_h=FORECAST_STEP_H,
    domain=(LON_MIN, LON_MAX, LAT_MIN, LAT_MAX),
    data_dir=DATA_DIR,
)
INIT_DATE = gfs_ds.attrs.get('init_date', 'unknown')
print(f'[TIME  ] {(_time.time() - _start):.1f} s')
print(f'[INIT  ] {INIT_DATE}')
print(f'[STEPS ] {gfs_ds.sizes["valid_time"]} lead times')
print(f'[VARS  ] {list(gfs_ds.data_vars)}')

## 1.2 Four-Panel Forecast Animation

The cell below assembles the forecast fields into an animated four-panel sequence displayed as a GIF.  Individual frames are also stored in outputs directory. 

In [ ]:
out_path = plot_four_panel_forecast_animation(
    gfs_ds,
    domain=(LON_MIN, LON_MAX, LAT_MIN, LAT_MAX),
    output_dir='outputs',
)
display(Image(filename=str(out_path)))
print(f'[OK]   Animation saved to {out_path}')

---

<div style="background-color: #fff3cd; padding: 15px; border-radius: 8px; border-left: 4px solid #ffc107;">

## 📝 Task 1.1 — Upper-Air Pattern Evolution

**Goal:** Assess the evolution of the four-panel upper-air pattern across the forecast horizon.

1. Examine the animation and describe how the dominant synoptic-scale feature in each panel evolves.

2. Based on the pattern evolution across all four panels, evaluate the potential for organized weather development over Greece across the forecast horizon.

The same assessment can be applied to any other country or region of interest within the analysis domain.

</div>

# Section 2: Ensemble Mean and Spread

The deterministic forecast in Section 1 provides a single estimate of the future atmospheric state, which cannot capture the range of plausible outcomes arising from imperfect initial conditions, model formulation, and other sources of error. Ensemble prediction addresses this by executing multiple integrations from slightly perturbed initial conditions, generating a population of forecast trajectories from which probabilistic information is extracted. The ensemble mean at each grid point estimates the expected atmospheric state, while the ensemble standard deviation quantifies the spread among members, with larger values indicating regions of higher forecast uncertainty.

Both quantities are computed for mean sea-level pressure (MSLP), a key field for diagnosing the position and intensity of surface pressure systems. Large ensemble spread in the MSLP field indicates significant uncertainty in the evolution of these synoptic features, with direct downstream implications for wind, temperature, and precipitation forecasts.

## 2.1 GEFS Ensemble Fields for MSLP

The cell below builds MSLP fields from all configured GEFS members across the full forecast horizon.

In [ ]:
gefs_mslp = build_gefs_forecast_da(
    _gefs_cycle,
    variable='mslp',
    domain=(LON_MIN, LON_MAX, LAT_MIN, LAT_MAX),
    horizon_h=FORECAST_HORIZON_H,
    data_dir=DATA_DIR,
    n_members=N_MEMBERS,
)
INIT_DATE = gefs_mslp.attrs.get('init_date', 'unknown')
print(f'[INIT  ] {INIT_DATE}')
print(f'[SHAPE ] {dict(gefs_mslp.sizes)}  (member × valid_time × latitude × longitude)')

## 2.2 Ensemble Mean and Standard Deviation

The cell below computes the ensemble mean and standard deviation of MSLP across the member dimension at each grid point and forecast hour.

In [ ]:
mslp_mean = gefs_mslp.mean(dim='member')
mslp_std  = gefs_mslp.std(dim='member')

print(f'[MEAN  ] min {float(mslp_mean.min()):.1f} hPa  max {float(mslp_mean.max()):.1f} hPa')
print(f'[STD   ] min {float(mslp_std.min()):.2f} hPa  max {float(mslp_std.max()):.2f} hPa')

## 2.3 Two-Panel Mean-and-Spread Map

The cell below maps the ensemble mean and standard deviation of MSLP at the longest available lead time, where forecast uncertainty is expected to be greatest. The left panel uses fixed-interval contours with labeled pressure centers, while the right panel shades the standard deviation against faint mean MSLP contours for spatial reference.

In [ ]:
mslp_mean_far = mslp_mean.sel(valid_time=FORECAST_HORIZON_H, method='nearest')
mslp_std_far  = mslp_std.sel(valid_time=FORECAST_HORIZON_H, method='nearest')

out_path = plot_ensemble_mean_spread(
    mslp_mean_far,
    mslp_std_far,
    domain=(LON_MIN, LON_MAX, LAT_MIN, LAT_MAX),
    lead_time_h=FORECAST_HORIZON_H,
    init_date=INIT_DATE,
    output_dir='outputs',
)
print(f'[OK]   Figure saved to {out_path}')

---

<div style="background-color: #fff3cd; padding: 15px; border-radius: 8px; border-left: 4px solid #ffc107;">

## 📝 Task 2.1 — Ensemble Forecast Uncertainty

**Goal:** Diagnose the spatial structure of ensemble forecast uncertainty from the MSLP mean and spread fields.

1. Identify the primary high and low pressure centers from the ensemble mean MSLP field (left panel).
2. Locate the regions of maximum and minimum forecast uncertainty using the ensemble standard deviation field (right panel).
3. Cross-reference both panels to determine which specific synoptic-scale features are associated with the greatest forecast uncertainty.

</div>

# Section 3: Point-Based Ensemble Forecast

A point-based forecast extracts the ensemble distribution at a single location, shifting the focus from the spatial structure of uncertainty to its temporal evolution at a specific site. In this section, an ensemble meteogram is constructed, presenting three surface variables across the configured forecast horizon: (i) air temperature at 2 m, (ii) 6-hour accumulated precipitation, and (iii) wind gust at 10 m.

## 3.1 Nearest grid point identification and surface variable retrieval

The cells below identify the nearest to the point of interest grid point and retrieve the full ensemble time series for the three targeted variables at that location.

In [ ]:
nearest_idx, distance_km = get_nearest_grid_point(
    poi_lat=POI_LAT,
    poi_lon=POI_LON,
    lat_grid=gefs_mslp.latitude.values,
    lon_grid=gefs_mslp.longitude.values,
)

nearest_lat = float(gefs_mslp.latitude.values[nearest_idx[0]])
nearest_lon = float(gefs_mslp.longitude.values[nearest_idx[1]])

print(
    f'[OK]   Nearest grid point: ({nearest_lat:.2f} N, {nearest_lon:.2f} E) — '
    f'{distance_km:.1f} km from {POI_NAME}'
)

In [ ]:
_build_kwargs = dict(
    domain=(LON_MIN, LON_MAX, LAT_MIN, LAT_MAX),
    horizon_h=FORECAST_HORIZON_H,
    data_dir=DATA_DIR,
    n_members=N_MEMBERS,
)

gefs_t2m    = build_gefs_forecast_da(_gefs_cycle, variable='t2m',       **_build_kwargs)
gefs_precip = build_gefs_forecast_da(_gefs_cycle, variable='precip_6h', **_build_kwargs)
gefs_gust   = build_gefs_forecast_da(_gefs_cycle, variable='gust10m',   **_build_kwargs)

t2m_point    = gefs_t2m.isel(   latitude=nearest_idx[0], longitude=nearest_idx[1])
precip_point = gefs_precip.isel(latitude=nearest_idx[0], longitude=nearest_idx[1])
gust_point   = gefs_gust.isel(  latitude=nearest_idx[0], longitude=nearest_idx[1])

print(f'[SHAPE ] T2m {dict(t2m_point.sizes)}  precip {dict(precip_point.sizes)}  gust {dict(gust_point.sizes)}')

## 3.2 Three-Panel Meteogram

The cell below constructs the three-panel meteogram.

The 2-meter temperature panel uses a plume diagram. All configures ensemble members are plotted as thin trajectories across the forecast horizon. The interquartile range and the 10th-to-90th percentile envelope are shaded with two tones. The control member is overlaid as a distinct line. Summary statistics annotated on the panel report the ensemble mean, standard deviation, and full range across the forecast horizon.

The precipitation panel uses 6-hour box plots at each forecast step. The box spans the 25th to 75th percentile of the member distribution, whiskers extend to the 10th and 90th percentiles, and outliers appear as individual points. For forecast hours where more than half of the members predict no precipitation, the box position reflects the zero-precipitation skew of the distribution.

The wind gust panel uses a plume diagram with the same percentile structure as the temperature panel. 

In [ ]:
out_path = build_meteogram(
    t2m=t2m_point,
    precip=precip_point,
    gust=gust_point,
    valid_times=gefs_t2m.valid_time.values,
    poi_name=POI_NAME,
    init_date=INIT_DATE,
    distance_km=distance_km,
    output_dir='outputs',
)

print(f'[OK]   Meteogram saved to {out_path}')

---

<div style="background-color: #fff3cd; padding: 15px; border-radius: 8px; border-left: 4px solid #ffc107;">

## 📝 Task 3.1 — Point Forecast Uncertainty

**Goal:** Characterize the temporal evolution of forecast uncertainty at the selected location from the three-panel meteogram.

1. Evaluate the expected weather conditions at the selected location and assess how forecast confidence changes over time.

2. Use plain language to communicate the forecast uncertainty at different horizons to a non-expert.

</div>

# Section 4: Weather Impact Assessment 

Probability maps quantify the fraction of ensemble members that satisfy a defined threshold at each grid point. A single-variable probability map is the primary product. The compound probability product extends this by combining several exceedance-fraction fields into a single map that highlights where multiple conditions required for a specific weather impact are simultaneously met.

The default case examined here is convection, where two variables used to define the thermodynamic environment favorable for organized storm development. Surface-based convective available potential energy (CAPE) above 1000 J/kg indicates sufficient instability to support moderate or greater thunderstorm development. Convective inhibition (CIN) above −60 J/kg indicates that the inhibiting cap is weak to moderate or absent, permitting most trigger mechanisms to initiate strom. The compound product is the element-wise multiplication of the two exceedance-fraction fields:

  $$\mathrm{Compound} = P(\mathrm{CAPE} > 1000) \cdot P(\mathrm{CIN} > -60)$$

This product serves as a probabilistic diagnostic, not a rigorous joint probability. Therefore, it is interpreted as a spatial indicator of where the two conditions share a meaningful likelihood across the ensemble, rather than the absolute probability of simultaneous occurrence. Where the compound product is high, the ensemble broadly agrees that conditions are favorable for severe weather; where it is low, at least one of the three conditions is absent from the majority of members.

> **Note on other impact scenarios**: The compound probability framework described above applies to all configured impact scenarios. Each case substitutes its own threshold variables and values, as specified in the configuration cell, while the exceedance-fraction computation and element-wise multiplication remain unchanged.

## 4.1 GEFS Member Fields for the Compound Probability Variables

The cell below retrieves GEFS ensemble members for the case-specific variables.

In [ ]:
_gefs_case = {}
for _var in _case_vars:
    _gefs_case[_var] = build_gefs_forecast_da(_gefs_cycle, variable=_var, **_build_kwargs)
    _unit = CASE_CFG['units'][_var]
    print(f'[{_var:<20}] min {float(_gefs_case[_var].min()):.2f}  max {float(_gefs_case[_var].max()):.2f} {_unit}')

## 4.2 Exceedance Fractions and Compound Product

The cell below computes the fraction of members exceeding each threshold at every grid point and forecast hour, and then derives the compound product as the element-wise multiplication of the exceedance-fraction fields.

In [ ]:
n_members = next(iter(_gefs_case.values())).sizes['member']

_prob = {}
for _var, _da in _gefs_case.items():
    _thr = CASE_CFG['thresholds'][_var]
    _prob[_var] = (_da > _thr).sum(dim='member') / n_members

# Compound product: element-wise multiplication of all exceedance fractions.
# Serves as a spatial diagnostic of joint condition alignment, not a rigorous
# joint probability.
_prob_list = list(_prob.values())
compound_prob = _prob_list[0]
for _p in _prob_list[1:]:
    compound_prob = compound_prob * _p

max_signal_idx    = int(compound_prob.max(dim=['latitude', 'longitude']).argmax(dim='valid_time'))
max_signal_lead_h = int(compound_prob.valid_time.values[max_signal_idx])

for _var, _p in _prob.items():
    _thr  = CASE_CFG['thresholds'][_var]
    _unit = CASE_CFG['units'][_var]
    print(f'[P({_var}>{_thr} {_unit})]  min {float(_p.min()):.3f}  |  max {float(_p.max()):.3f}')
print(f'[COMPOUND]  max {float(compound_prob.max()):.4f}  at lead {max_signal_lead_h} h')

## 4.3 Map the compound probability field

The cell below maps the compound probability at the lead time of maximum signal: compound product shaded with a perceptually uniform colormap, with the three individual exceedance-fraction fields overlaid as contours at a common probability level in three distinct colors. An animated GIF is exported across the full forecast horizon.

In [ ]:
out_path = plot_compound_probability_animation(
    compound_prob=compound_prob,
    domain=(LON_MIN, LON_MAX, LAT_MIN, LAT_MAX),
    init_date=INIT_DATE,
    max_signal_lead_h=max_signal_lead_h,
    case=ACTIVE_CASE,
    output_dir='outputs',
)
display(Image(filename=str(out_path)))
print(f'[OK]   Animation saved to {out_path}')

---

<div style="background-color: #fff3cd; padding: 15px; border-radius: 8px; border-left: 4px solid #ffc107;">

## 📝 Task 4.1 — Compound Probability Interpretation

**Goal:** Interpret the compound probability map in the context of the synoptic-scale forecast.

1. Identify the region of highest compound probability and the forecast hour at which the signal peaks.

2. Cross-reference the compound probability map with the four-panel animation from Section 1. Using the synoptic-scale pattern at the relevant forecast hour, characterize the atmospheric conditions over the identified region and assess the potential for organized convective weather development.

3. Communicate the expected  weather impact to a non-expert. What type of weather is likely, where, and how confident is the forecast.

</div>

<div style="background-color: #f8d7da; padding: 15px; border-radius: 8px; border-left: 4px solid #dc3545;">

## Pro Task: Compound Probability — Alternative Case

**Goal:** Repeat the compound probability analysis for an alternative event and initialization.

**Instructions:**

Return to the **Configuration** cell, change the `ACTIVE_CASE` and `TARGET_DATE`, and re-run all cells. Repeat the analysis for any of the following events:

1. **Heat Wave Kleon** (23 Jul 2023): An extended Mediterranean heat wave affecting Greece and the central Mediterranean.
2. **Storm Ciarán** (2 Nov 2023): An exceptionally deep and rapidly intensifying extratropical cyclone over Western Europe.
3. **Cold outbreak Ariande** (10 Jan 2017): Continental Arctic air mass intrusion with subzero minimum temperature and widespread snow cover over the Balkan peninsula and Greece.

</div>

## 🏁 Summary & Key Takeaways

The forecasting workflow applied in this notebook advances from upper-air deterministic forecasts through ensemble mean and spread diagnostics to probabilistic products, drawing on GFS and GEFS output for the European domain.

**Key Takeaways:**

* ✅ **Upper-Air Deterministic Forecast:** The GFS four-panel animation tracks the forecast evolution of 850 hPa thermal structure, 700 hPa moisture, 500 hPa geopotential height, and 250 hPa wind speed across the configured horizon.
* ✅ **Ensemble Mean and Spread:** At mean sea-level pressure, the ensemble mean and spread diagnose the expected synoptic-scale pressure pattern and identify the regions where forecast uncertainty is greatest.
* ✅ **Point-Based Ensemble Forecast:** At point level, ensemble plumes and precipitation box plots resolve the temporal evolution of forecast uncertainty for near-surface air temperature, accumulated precipitation, and wind gust.
* ✅ **Weather Impact Assessment:** The compound probability product maps where all case-specific exceedance conditions are simultaneously met across the ensemble, providing a spatially explicit diagnostic of high-impact weather potential.

---

**🦉 Crafted with wisdom at One Weather Lab (OWL)**<br>
Laboratory of Meteorology and Climatology, Physics Department, University of Ioannina<br>
Christos Giannaros <<chris.giannaros@uoi.gr>>